# 🔥 Fine-Tune Gemma 4 on Human Emotions Dataset

This notebook fine-tunes **Gemma 4 E4B-it** on the [`dair-ai/emotion`](https://huggingface.co/datasets/dair-ai/emotion) dataset using **LoRA + 4-bit quantization** — designed to run on a free/paid Google Colab GPU (T4 or A100).

**Task:** Classify text into one of 6 emotions: `sadness`, `joy`, `love`, `anger`, `fear`, `surprise`

---
### Before You Start
1. Go to **Runtime → Change runtime type → GPU** (T4 is free, A100 is faster)
2. You need a **Hugging Face token** with access to the gated `google/gemma-4-E4B-it` model
   - Get it at: https://huggingface.co/settings/tokens
3. Add your token to **Colab Secrets** (`🔑` icon on left sidebar) as `HF_TOKEN`

## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install -U transformers accelerate datasets trl peft bitsandbytes scikit-learn huggingface_hub

## Step 2 — Authenticate with Hugging Face

> 💡 Store your token in **Colab Secrets** (key icon in left sidebar) as `HF_TOKEN` — do NOT paste it directly in code.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("Logged in to Hugging Face.")

✅ Logged in to Hugging Face.


## Step 3 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu}")
    print(f"   VRAM: {vram:.1f} GB")
    # Recommend batch size based on GPU
    if vram < 16:
        print("⚠️  Less than 16GB VRAM detected. Using smaller batch size (2). Training will be slower.")
        BATCH_SIZE = 2
    elif vram < 40:
        print("ℹ️  ~16GB VRAM (T4/3090). Using batch size 4.")
        BATCH_SIZE = 4
    else:
        print("🚀 High VRAM GPU (A100/H100). Using batch size 8.")
        BATCH_SIZE = 8
else:
    raise RuntimeError("❌ No GPU found! Go to Runtime → Change runtime type → GPU")

✅ GPU: Tesla T4
   VRAM: 15.6 GB
⚠️  Less than 16GB VRAM detected. Using smaller batch size (2). Training will be slower.


## Step 4 — Load the Emotion Dataset

In [ ]:
from datasets import load_dataset, DatasetDict

# Limits to keep training fast on Colab — increase for better results
TRAIN_LIMIT      = 4000
VALIDATION_LIMIT = 400
TEST_LIMIT       = 400
EVAL_LIMIT       = 400

raw_dataset = load_dataset("dair-ai/emotion")

def maybe_limit(split, limit):
    split = split.shuffle(seed=42)
    return split.select(range(min(limit, len(split)))) if limit else split

dataset = DatasetDict({
    "train":      maybe_limit(raw_dataset["train"],      TRAIN_LIMIT),
    "validation": maybe_limit(raw_dataset["validation"], VALIDATION_LIMIT),
    "test":       maybe_limit(raw_dataset["test"],       TEST_LIMIT),
})

label_names = dataset["train"].features["label"].names

print("Dataset splits:")
print(dataset)
print(f"\nEmotion labels: {label_names}")
print(f"\nExample: {dataset['train'][0]}")

Dataset splits:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 4000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 400
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 400
    })
})

Emotion labels: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

Example: {'text': 'while cycling in the country', 'label': 4}


## Step 5 — Format Data for Gemma 4 Chat Format

Gemma 4 expects a **system → user → assistant** chat structure for supervised fine-tuning.

In [ ]:
SYSTEM_PROMPT = """You are an emotion classification assistant.
Read the user's text and answer with exactly one label.
Only choose from: sadness, joy, love, anger, fear, surprise.
Return only the label and nothing else."""

def to_prompt_completion(example):
    text  = example["text"]
    label = label_names[example["label"]]
    return {
        "prompt": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": f"Classify the emotion of this text:\n\n{text}"},
        ],
        "completion": [
            {"role": "assistant", "content": label}
        ],
    }

sft_dataset = dataset.map(
    to_prompt_completion,
    remove_columns=dataset["train"].column_names
)

print("✅ Dataset formatted for SFT.")
print("\nFormatted example:")
print(sft_dataset["train"][0])

✅ Dataset formatted for SFT.

Formatted example:
{'prompt': [{'role': 'system', 'content': "You are an emotion classification assistant.\nRead the user's text and answer with exactly one label.\nOnly choose from: sadness, joy, love, anger, fear, surprise.\nReturn only the label and nothing else."}, {'role': 'user', 'content': 'Classify the emotion of this text:\n\nwhile cycling in the country'}], 'completion': [{'role': 'assistant', 'content': 'fear'}]}


## Step 6 — Load Gemma 4 E4B-it with 4-bit Quantization

> ⏳ This will download ~4GB of model weights. Takes 3–5 minutes on Colab.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID   = "google/gemma-4-E4B-it"
MODEL_DTYPE = torch.bfloat16

# Enable TF32 for faster matmul on Ampere+ GPUs
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Load tokenizer
processor = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if processor.pad_token is None:
    processor.pad_token = processor.eos_token

# 4-bit quantization config (reduces ~16GB model → ~4GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=MODEL_DTYPE,
)

# Load model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

# Align special token IDs
for cfg in [base_model.config, base_model.generation_config]:
    cfg.pad_token_id = processor.pad_token_id
    cfg.bos_token_id = processor.bos_token_id
    cfg.eos_token_id = processor.eos_token_id
base_model.config.use_cache = False

print("✅ Gemma 4 E4B-it loaded with 4-bit quantization.")

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

✅ Gemma 4 E4B-it loaded with 4-bit quantization.


## Step 7 — Baseline Evaluation (Before Fine-Tuning)

We evaluate the base model first so we can compare improvement after training.

In [ ]:
import re
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix

LABEL_PATTERN = re.compile(r"\b(sadness|joy|love|anger|fear|surprise)\b", re.IGNORECASE)
VALID_LABELS  = set(label_names)

def extract_label(raw_text: str) -> str:
    raw_text = raw_text.strip().lower()
    match = LABEL_PATTERN.search(raw_text)
    if match:
        return match.group(1)
    tokens = raw_text.split()
    return tokens[0].strip(".,!?:;\"'()[]{}") if tokens else ""

def generate_label(model, proc, user_text, max_new_tokens=4):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Classify the emotion of this text:\n\n{user_text}"},
    ]
    device = next(model.parameters()).device
    inputs = proc.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt"
    ).to(device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=proc.pad_token_id,
            eos_token_id=proc.eos_token_id,
        )
    raw = proc.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return extract_label(raw)

def evaluate_model(model, proc, split="test", limit=EVAL_LIMIT):
    y_true, y_pred, rows = [], [], []
    source = dataset[split]
    if limit:
        source = source.select(range(min(limit, len(source))))
    model.eval()
    for ex in tqdm(source, desc=f"Evaluating {split}"):
        true_label = label_names[ex["label"]]
        raw_pred   = generate_label(model, proc, ex["text"])
        pred_label = raw_pred if raw_pred in VALID_LABELS else "INVALID"
        y_true.append(true_label)
        y_pred.append(pred_label)
        rows.append({"text": ex["text"], "true": true_label,
                     "pred": pred_label, "correct": true_label == pred_label})
    metrics = {
        "accuracy":    accuracy_score(y_true, y_pred),
        "macro_f1":    f1_score(y_true, y_pred, labels=label_names, average="macro", zero_division=0),
        "invalid":     sum(1 for p in y_pred if p == "INVALID"),
        "n_evaluated": len(y_true),
    }
    return metrics, pd.DataFrame(rows)

# Quick single-example sanity check
sample = generate_label(base_model, processor, "I feel so happy and excited today!")
print(f"Quick test → 'I feel so happy and excited today!' → {sample}")

# Full baseline evaluation
pre_metrics, pre_df = evaluate_model(base_model, processor)
print("\n📊 Baseline Results (before fine-tuning):")
for k, v in pre_metrics.items():
    print(f"   {k}: {v}")

Quick test → 'I feel so happy and excited today!' → joy


Evaluating test:   0%|          | 0/400 [00:00<?, ?it/s]


📊 Baseline Results (before fine-tuning):
   accuracy: 0.58
   macro_f1: 0.42237965844124475
   invalid: 33
   n_evaluated: 400


## Step 8 — Fine-Tune with LoRA

We use **LoRA (rank=16)** to attach lightweight trainable adapters on top of the frozen base model. Only ~0.5% of parameters are updated — making this feasible on a single Colab GPU.

> ⏳ Training ~4000 examples for 1 epoch takes ~8–15 min on T4, ~4–6 min on A100.

In [ ]:
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

# Unload any existing LoRA adapters if re-running this cell
if isinstance(base_model, PeftModel):
    base_model = base_model.unload()
    base_model.config.use_cache = False

# ── LoRA config ──────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

# ── Training config ───────────────────────────────────────────────────────────
training_args = SFTConfig(
    output_dir="./gemma4-emotion-lora",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=50,
    num_train_epochs=1,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_checkpointing=True,
    bf16=True,
    fp16=False,
    tf32=False,
    max_length=256,
    packing=False,
    completion_only_loss=True,
    remove_unused_columns=False,
    dataloader_num_workers=2,
    optim="paged_adamw_8bit",
    report_to="none",
)

# ── Trainer setup ─────────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=base_model,
    train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["validation"],
    peft_config=lora_config,
    args=training_args,
    processing_class=processor,
)

# Verify LoRA params were attached
trainable_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
if trainable_params == 0:
    raise RuntimeError("No trainable LoRA parameters found. Check target_modules.")
print(f"🔧 Trainable LoRA parameters: {trainable_params:,}")

# ── Train! ────────────────────────────────────────────────────────────────────
print("\n🚀 Starting training...")
train_result = trainer.train()
trainer.model.eval()
trainer.model.config.use_cache = True

print("\n✅ Training complete!")
print(train_result)


🔧 Trainable LoRA parameters: 50,499,584

🚀 Starting training...


Step,Training Loss,Validation Loss
100,0.400324,0.256330
200,0.291718,0.273964
300,0.164552,0.178069
400,0.135828,0.161694
500,0.136186,0.128060
600,0.135405,0.114716
700,0.091282,0.116037
800,0.069742,0.129889
900,0.136028,0.087239
1000,0.055002,0.084602



✅ Training complete!
TrainOutput(global_step=1000, training_loss=0.20491755437850953, metrics={'train_runtime': 9748.1005, 'train_samples_per_second': 0.41, 'train_steps_per_second': 0.103, 'total_flos': 1.0698120875155968e+16, 'train_loss': 0.20491755437850953})


## Step 9 — Save the Fine-Tuned Model

In [ ]:
SAVE_DIR = "./gemma4-emotion-lora"

trainer.model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

print(f"✅ Model and tokenizer saved to {SAVE_DIR}")

✅ Model and tokenizer saved to ./gemma4-emotion-lora


## Step 10 — Evaluate the Fine-Tuned Model

In [ ]:
post_metrics, post_df = evaluate_model(trainer.model, processor)

print("\n📊 Results Comparison:")
print(f"{'Metric':<20} {'Before':>10} {'After':>10} {'Change':>10}")
print("-" * 52)
for key in ["accuracy", "macro_f1"]:
    before = pre_metrics[key]
    after  = post_metrics[key]
    delta  = after - before
    arrow  = "▲" if delta > 0 else "▼"
    print(f"{key:<20} {before:>10.4f} {after:>10.4f} {arrow}{abs(delta):>9.4f}")
print(f"{'invalid_preds':<20} {pre_metrics['invalid']:>10} {post_metrics['invalid']:>10}")

print(f"\n🎯 Accuracy: {pre_metrics['accuracy']:.1%} → {post_metrics['accuracy']:.1%}")
print(f"🎯 Macro F1:  {pre_metrics['macro_f1']:.4f} → {post_metrics['macro_f1']:.4f}")

Evaluating test:   0%|          | 0/400 [00:00<?, ?it/s]


📊 Results Comparison:
Metric                   Before      After     Change
----------------------------------------------------
accuracy                 0.5800     0.9200 ▲   0.3400
macro_f1                 0.4224     0.8754 ▲   0.4530
invalid_preds                33          0

🎯 Accuracy: 58.0% → 92.0%
🎯 Macro F1:  0.4224 → 0.8754


## Step 11 — Full Classification Report

In [ ]:
from sklearn.metrics import classification_report

print("Classification Report (Fine-tuned model):")
print(classification_report(
    post_df["true"], post_df["pred"],
    labels=label_names, zero_division=0
))

Classification Report (Fine-tuned model):
              precision    recall  f1-score   support

     sadness       0.92      0.96      0.94       114
         joy       0.95      0.95      0.95       162
        love       0.83      0.73      0.78        26
       anger       0.93      0.89      0.91        47
        fear       0.87      0.92      0.89        36
    surprise       0.85      0.73      0.79        15

    accuracy                           0.92       400
   macro avg       0.89      0.86      0.88       400
weighted avg       0.92      0.92      0.92       400



## Step 12 — Try Your Own Text!

In [ ]:
test_texts = [
    "I can't believe how amazing this day has been!",
    "I'm so frustrated and angry right now.",
    "I miss you so much and feel so alone.",
    "Oh my god, I didn't see that coming at all!",
    "I deeply care for you and want the best for you.",
    "Something feels very wrong and I'm scared.",
]

print("🧪 Inference on custom examples:\n")
for text in test_texts:
    label = generate_label(trainer.model, processor, text)
    print(f"  '{text}'")
    print(f"   → {label.upper()}\n")

🧪 Inference on custom examples:

  'I can't believe how amazing this day has been!'
   → JOY

  'I'm so frustrated and angry right now.'
   → ANGER

  'I miss you so much and feel so alone.'
   → SADNESS

  'Oh my god, I didn't see that coming at all!'
   → SURPRISE

  'I deeply care for you and want the best for you.'
   → LOVE

  'Something feels very wrong and I'm scared.'
   → SADNESS



## (Optional) Step 13 — Push to Hugging Face Hub

Upload your fine-tuned adapter to the Hub so you can reuse or share it.

In [ ]:
# Change this to your HF username/repo
HF_REPO_ID = "najus/gemma4-emotion-lora"  # ← EDIT THIS

trainer.model.push_to_hub(HF_REPO_ID, private=False)
processor.push_to_hub(HF_REPO_ID, private=False)

print(f"✅ Model pushed to https://huggingface.co/{HF_REPO_ID}")

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   2%|1         | 3.39MB /  202MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp4ij5azo2/tokenizer.json:  74%|#######4  | 23.9MB / 32.2MB            

✅ Model pushed to https://huggingface.co/najus/gemma4-emotion-lora


---
## 📝 Notes

| Setting | Value | Why |
|---------|-------|-----|
| Quantization | 4-bit NF4 | Reduces model from ~16GB → ~4GB VRAM |
| LoRA rank | 16 | Good balance of capacity vs memory |
| Compute dtype | bfloat16 | More stable than float16 for training |
| Optimizer | paged_adamw_8bit | Offloads optimizer states, saves VRAM |
| Train examples | 4,000 | Fast to train; increase for better results |
| Epochs | 1 | Prevents overfitting on small dataset |

**To improve accuracy further:**
- Increase `TRAIN_LIMIT` to the full ~16,000 training examples
- Train for 2–3 epochs
- Increase LoRA rank to 32 or 64
- Use a larger Gemma 4 variant (requires more VRAM)